# Tarea 25 · Regresión lineal del cargo acumulado

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/feat/25-regresion-lineal/notebooks/trabajo/25_regresion_lineal.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable:** Julián.

Estimamos `TotalCharges` a partir de `tenure` con una regresión lineal simple. La entrada es `data/processed/telco_clean.csv`. El resultado es el modelo ajustado y su evaluación con R² y MAE sobre datos de prueba.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt`, abrir el notebook dentro del repositorio y ejecutar todas las celdas.
- **Colab:** abrir el botón tras publicar la rama o subir este notebook. El CSV se lee desde GitHub.
- Al integrar la tarea, cambiar el enlace del botón a `main`.

## 1. Carga de datos

Usamos el dataset limpio del punto 9. Revisamos que ambas variables sean numéricas, finitas y completas antes de ajustar el modelo.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

ruta_datos = Path("data/processed/telco_clean.csv")
if EN_COLAB:
    fuente = (
        "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
        + ruta_datos.as_posix()
    )
else:
    actual = Path.cwd().resolve()
    fuente = next(
        (base / ruta_datos for base in (actual, *actual.parents)
         if (base / ruta_datos).is_file()),
        None,
    )
    if fuente is None:
        raise FileNotFoundError("Falta telco_clean.csv. Ejecutá primero la tarea 9.")

datos = pd.read_csv(fuente, usecols=["tenure", "TotalCharges"])
assert all(pd.api.types.is_numeric_dtype(datos[columna]) for columna in datos)
assert np.isfinite(datos.to_numpy()).all(), "Revisá faltantes o valores infinitos."
print(f"Fuente: {fuente}")
print(f"Clientes: {len(datos):,}")

## 2. Partición y ajuste

El predictor es la antigüedad en meses y el objetivo es el cargo acumulado. Reservamos un 20 % de los clientes para evaluar el modelo y entrenamos con el otro 80 %, usando `random_state=42` para reproducir la división. Esta tarea tiene su propia partición, como indica el plan.

El modelo ajusta una recta con la forma cargo estimado = intercepto + coeficiente × antigüedad. Aprende los dos parámetros únicamente con entrenamiento.

In [ ]:
X = datos[["tenure"]]
y = datos["TotalCharges"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
coeficiente = modelo.coef_[0]
intercepto = modelo.intercept_

print(f"Entrenamiento: {len(X_train):,} clientes")
print(f"Prueba: {len(X_test):,} clientes")
print(f"Cargo estimado = {intercepto:.2f} + {coeficiente:.2f} × antigüedad")

## 3. Evaluación sobre prueba

R² compara los errores cuadrados del modelo con la variación del objetivo en prueba. Un valor de 1 corresponde a un ajuste perfecto y puede ser negativo si los errores son grandes. MAE es el promedio de las diferencias absolutas entre los cargos observados y los estimados, expresado en las mismas unidades monetarias que el cargo acumulado.

In [ ]:
predicciones = modelo.predict(X_test)
r2 = r2_score(y_test, predicciones)
mae = mean_absolute_error(y_test, predicciones)

metricas = pd.DataFrame({
    "Métrica": ["R² en prueba", "MAE en prueba"],
    "Valor": [r2, mae],
})
print(metricas.to_string(index=False, float_format=lambda valor: f"{valor:.4f}"))

## 4. Interpretación

Interpretamos los resultados impresos por las celdas anteriores. El coeficiente describe una asociación promedio entre la antigüedad y el total facturado en esta muestra. Los clientes tienen distintos planes y cargos mensuales, por lo que la antigüedad por sí sola deja parte de la variación sin explicar.

In [ ]:
print(
    f"Por cada mes adicional de antigüedad, la recta estima "
    f"{coeficiente:.2f} unidades monetarias más de cargo acumulado."
)
print(
    f"En los clientes de prueba, R² = {r2:.3f}. "
    f"El error absoluto promedio es de {mae:.2f} unidades monetarias."
)
print(
    f"El intercepto es {intercepto:.2f}: es el valor de la recta a cero meses. "
    "Un intercepto negativo carece de interpretación como cargo real "
    "y muestra una limitación del ajuste lineal cerca del origen."
)